### RAG Pipeline Modular Coding

In [20]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain.schema import Document
from langchain.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain.schema.runnable import RunnableLambda, RunnableMap
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [21]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [22]:
# Custom Semantic Chunker w Threshold

class ThresholdSemanticChunker:
    def __init__(self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model=SentenceTransformer(model_name)
        self.threshold=threshold

    def split(self,text:str):
        sentences=[s.strip() for s in text.split(".") if s.strip()]
        embeddings=self.model.encode(sentences) #Sentences transformer, converts to vectors
        chunks=[]
        current_chunk=[sentences[0]]

        for i in range(1,len(sentences)):
            sim=cosine_similarity([embeddings[i-1]],[embeddings[i]])[0][0]
            if sim>=self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk)+".")
                current_chunk=[sentences[i]]
        chunks.append(". ".join(current_chunk)+".")
        return chunks
    
    def split_documents(self,docs):
        result=[]
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk,metadata=doc.metadata))
        
        return result



In [23]:
# Sample text
sample_text = """
The weather is too hot today. I feel thirsty. I am bored because I have nothing to do.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nThe weather is too hot today. I feel thirsty. I am bored because I have nothing to do.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

### Chunking

In [24]:
#Chunking

chunker=ThresholdSemanticChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

[Document(metadata={}, page_content='The weather is too hot today.'),
 Document(metadata={}, page_content='I feel thirsty.'),
 Document(metadata={}, page_content='I am bored because I have nothing to do.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

### VectorStore

In [25]:
embedding_model=HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
vectorstore=FAISS.from_documents(chunks,embedding_model)
retriever=vectorstore.as_retriever()

You actually used embeddings two times — but for two different purposes.
1) First time (inside your Chunker class)

Purpose: To decide how to split the text.

embeddings = self.model.encode(sentences)


Here we are only comparing similarity between neighboring sentences to see:

Do these sentences belong together?

Or should we start a new chunk?

These embeddings are temporary and thrown away after chunking.

We never store them.
We never search them.
We only use them to determine grouping.

This embedding = only used for deciding chunk boundaries

2) Second time (before storing in FAISS)

Purpose: To store the final chunks so we can search later.

vectorstore = FAISS.from_documents(chunks, embedding_model)


Here the embeddings are created again — but this time one vector per final chunk, and these are stored in FAISS for retrieval.

This embedding = used to search and retrieve relevant chunks later

### Prompt Template

In [26]:
template=""" 
Answer the question based on the following context:
{context}

Question: {question}
"""

prompt=PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template=' \nAnswer the question based on the following context:\n{context}\n\nQuestion: {question}\n')

### LLM

In [27]:
llm=init_chat_model(model="groq:llama-3.1-8b-instant",temperature=0.4)

If the temperature is low (like 0 or 0.2):

The model becomes more logical, accurate, and predictable

It will try to give the most correct / safest answer every time

Good for: QA, RAG, coding, factual answers

Low temperature = serious + consistent

If the temperature is high (like 0.7 to 1.2):

The model becomes more random, creative, and expressive

It may produce new ideas, but sometimes mistakes

Good for: story writing, brainstorming, creative chat

High temperature = creative + varied

### LCEL Chain with Retrieval

In [31]:
rag_chain=(
    RunnableMap(
        {
            "context": lambda x:"\n\n".join(
                [doc.page_content for doc in retriever.invoke(x['question'])]
            ),
            "question":lambda x:x["question"]
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

### Query

In [32]:
query={"question":"Why am i bored?"}
result=rag_chain.invoke(query)

print(result)

You're bored because you have nothing to do.


1. RunnableMap({ ... })
RunnableMap(
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"]
    }
)


This takes the model input (example: {"question": "Where is the Eiffel Tower?"}),
and creates two things:

Output Key	What it returns	Meaning
"context"	retrieved text chunks	The info to help answer the question
"question"	original question	So the prompt still has the user's question
Let’s decode the lambdas:

context:

lambda x: retriever.invoke(x["question"])


→ Take the user’s question
→ Use retriever to pull relevant chunks
→ Return those chunks

question:

lambda x: x["question"]


→ Just return the question as it is

So after this step, our input becomes:

{
  "context": "The Eiffel Tower is in Paris.",
  "question": "Where is the Eiffel Tower?"
}

2. | prompt
| prompt


This injects context and question into your prompt template.

The prompt receives data like:

Context: The Eiffel Tower is in Paris.
Question: Where is the Eiffel Tower?
Answer:


The LLM now knows what to refer to.

3. | llm
| llm


This sends the formatted prompt to your Groq LLM.

The LLM generates an answer (string-like chat response).

Example output:

"The Eiffel Tower is located in Paris, France."

4. | StrOutputParser()
| StrOutputParser()


This simply converts the LLM output into a clean string, removing ChatMessage objects or metadata.

So final output becomes a plain text answer: